# MovieLens latest-small 探索的データ分析

## 目的

MovieLens latest-smallの評価データと映画データを確認し、映画推薦システムの設計に必要な特徴や問題点を把握する。

## このNotebookで確認する内容

- ユーザー数、映画数、評価数
- 評価値の分布
- ユーザーごとの評価件数
- 映画ごとの評価件数
- 評価件数が少ない映画の割合
- ジャンルごとの映画数
- 評価データの疎らさ

> 現在は最初の作業単位として、データの読み込みと構造確認までを実装する。

## 1. ライブラリの読み込み

- `pathlib.Path`：ファイルやフォルダのパスを扱う
- `pandas`：CSVをDataFrameとして読み込み、表形式データを操作する
- `matplotlib.pyplot`：グラフを作成する（後のEDAで使用）

In [4]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

## 2. データファイルの場所を確認する

Notebookを`notebooks`フォルダから実行した場合と、プロジェクト直下から実行した場合の両方に対応して、プロジェクトの基準位置を決める。

`Path.exists()`は、指定したファイルが存在すれば`True`、存在しなければ`False`を返す。

In [5]:
# Notebookの現在の実行位置を確認する
current_dir = Path.cwd()

# notebooksフォルダで実行している場合は、1つ上をプロジェクトの基準位置にする
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_dir = project_root / "data" / "raw" / "ml-latest-small"
ratings_path = data_dir / "ratings.csv"
movies_path = data_dir / "movies.csv"

print("現在の実行位置:", current_dir)
print("データフォルダ:", data_dir)
print("ratings.csvが存在するか:", ratings_path.exists())
print("movies.csvが存在するか:", movies_path.exists())

現在の実行位置: /Users/ma-ya/Free/movie-recommendation-system/notebooks
データフォルダ: /Users/ma-ya/Free/movie-recommendation-system/data/raw/ml-latest-small
ratings.csvが存在するか: True
movies.csvが存在するか: True


## 3. CSVをDataFrameとして読み込む

`pd.read_csv()`はCSVファイルのパスを入力として受け取り、表形式の`DataFrame`を返す。

データがまだ配置されていない場合は、期待する配置先を示すエラーで処理を止める。

In [6]:
# 必要なCSVが両方そろっていることを読み込み前に確認する
if not ratings_path.exists() or not movies_path.exists():
    raise FileNotFoundError(
        "MovieLensのCSVが見つかりません。"
        f"{data_dir} にratings.csvとmovies.csvを配置してください。"
    )

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print("CSVの読み込みが完了しました。")

CSVの読み込みが完了しました。


## 4. 読み込み結果を確認する

- `DataFrame.shape`：`(行数, 列数)`を返す
- `DataFrame.head()`：先頭5行をDataFrameとして返す
- `display()`：DataFrameをNotebook上で表として表示する

In [7]:
print("ratingsの形:", ratings.shape)
print("moviesの形:", movies.shape)

print("ratingsの先頭5行")
display(ratings.head())

print("moviesの先頭5行")
display(movies.head())

ratingsの形: (100836, 4)
moviesの形: (9742, 3)
ratingsの先頭5行


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


moviesの先頭5行


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## この時点で確認すること

- `ratings.csv`と`movies.csv`がどちらも見つかるか
- `ratings`がおよそ100,836行・4列か
- `movies`がおよそ9,742行・3列か
- `userId`、`movieId`、`rating`、`timestamp`の意味を説明できるか
- `movieId`、`title`、`genres`の意味を説明できるか

次の作業単位では、欠損値、重複、データ型、評価値の範囲を確認する。

## データ構造の確認結果

- `ratings.csv`と`movies.csv`が存在し、想定した行数・列数で読み込めた。現時点では「ファイルが破損していない」と断定するのではなく、「想定した形式で読み込めた」と表現する。
- `userId`は、評価を行ったユーザーを識別する匿名IDである。
- `movieId`は映画を識別するIDであり、`ratings`と`movies`を結合する共通キーである。
- `rating`はユーザーが映画に付けた評価値である。先頭行では4.0や5.0を確認できたが、データ全体が想定範囲内かは、最小値・最大値・固有値を集計して確認する必要がある。
- `timestamp`は評価された日時を表し、過去の評価を学習用、未来の評価を評価用にする時系列分割で利用できる。
- `title`は映画名であり、主に推薦結果を人が理解できる形で表示するために利用する。映画の識別には`movieId`を使用する。
- `genres`は映画の大まかな特徴を表す。今回は主推薦方式には直接使用しないが、EDA、推薦結果の解釈、改善案の検討に利用できる。

### 現時点では未確認の内容

行数と列数が想定どおりでも、各セルの欠損値、重複、範囲外の評価値、`ratings`にしか存在しない`movieId`が含まれている可能性は残る。これらは次の分析で個別に確認する。